# 📗 Cypher 기초: 조건·패턴·결과 다듬기·멱등 적재

앞서 `CREATE` 로 만들고 `MATCH`·`RETURN` 으로 찾고 `SET`·`REMOVE` 로 고치는 법을 배웠습니다. 이번 시간에는 **WHERE** 로 비교·조합·부정 조건을 걸고, 여러 노드를 잇는 **다단 관계 패턴**을 읽고, 돌려받은 결과를 **DISTINCT·ORDER BY·LIMIT** 로 다듬고, **MERGE** 로 "있으면 그대로, 없으면 만들기"(멱등)를 익힙니다.

## ⏪ 복습: 지난 시간까지

- **CREATE** 로 노드 `(:레이블 {속성})` 와 관계 `(a)-[:종류]->(b)` 를 만들었습니다. 관계에도 속성을 붙였고(`-[:종류 {속성: 값}]->`), 관계를 변수에 담아(`-[r:종류]->`) 그 값을 꺼냈습니다.
- **MATCH … RETURN … AS 별칭** 으로 찾아서 값을 돌려받았습니다. `CREATE … RETURN` 으로 방금 만든 것을 그 자리에서 받을 수도 있었죠.
- **SET** 으로 이미 있는 노드의 속성을 고치거나 새로 더했고, **REMOVE** 로 속성·레이블을 뗐습니다.
- **DELETE·DETACH DELETE** 로 잘못 만든 관계·노드를 지웠습니다.
- 패턴 안 **속성 map** `{name: '김서준'}` 으로 범위를 좁혔습니다. 오늘은 WHERE 로 더 넓힙니다.

**오늘의 목표**

**1. 조건으로 거르기**
- [ ] (1-1) **WHERE** 로 비교(`>=`, `<=`, `<>`) 조건을 건다.
- [ ] (1-2) **AND / OR** 로 여러 조건을 조합한다.
- [ ] (1-3) **NOT** 으로 조건을 뒤집고, 속성이 비어 있는 노드를 **IS NULL** 로 찾는다.

**2. 여러 노드를 잇는 패턴**
- [ ] (2-1) **체인**과 **역방향**을 읽고 쓴다.
- [ ] (2-2) **공유 패턴**으로 가운데 노드를 함께 가리킨다.

**3. 결과 다듬기**
- [ ] (3-1) 결과가 **패턴이 맞은 경우의 수**만큼 나온다는 것을 안다.
- [ ] (3-2) **RETURN DISTINCT** 로 되풀이되는 행을 접는다.
- [ ] (3-3) **ORDER BY** 로 줄을 세우고 **LIMIT** 으로 위에서 몇 개만 받는다.

**4. 멱등 적재**
- [ ] (4-1) **MERGE** 로 노드와 관계를 멱등하게 적재한다.
- [ ] (4-2) MERGE 가 무엇을 "같은 것" 으로 보는지 알고 **식별 속성만** 적는다.
- [ ] (4-3) **ON CREATE SET · ON MATCH SET** 으로 처음 만들 때와 다시 만났을 때를 가른다.

**5. 값 넘기기**
- [ ] 파이썬 값을 **`$이름` 자리표시자**로 넘긴다(문자열에 이어 붙이지 않는다).

아래 준비 셀 4개를 위에서부터 실행하세요.

**연결이 안 되면**: `.env` 의 접속 정보가 지금 켜져 있는 실습용 데이터베이스의 값인지 먼저 확인하세요. `ServiceUnavailable` 은 **DB 가 꺼져 있거나 포트가 다른 것**, `AuthError` 는 **아이디·비밀번호가 다른 것**입니다.

In [1]:
# [제공 코드] Neo4j 연결: 실행만 하세요. 반드시 "실습 전용" DB 여야 합니다(아래 실습이 그래프를 지웁니다).
# 앞으로 모든 Cypher 는 run_cypher("쿼리", 파라미터=값) 으로 실행하고, 결과는 dict 리스트로 옵니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # 지우기 규칙 위반 에러

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env", override=True)       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러 없이 기본값으로 넘어간다. 마지막 줄에 찍히는 주소를 눈으로 꼭 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 여기까지 찍히면 준비 완료

Neo4j 연결: bolt://localhost:7687


> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 지난 단원에서 브라우저로 적재한 **Movies 예제 그래프와 그때 푼 과제 결과도 함께 사라집니다.** 되돌릴 수 없으니, `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요. Movies 를 남기고 싶다면 실습용 인스턴스를 따로 하나 만들어 그 접속 정보를 `.env` 에 넣으면 됩니다(지웠더라도 day28 폴더의 `data/movies_setup.cypher` 로 다시 적재할 수 있습니다).

In [2]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙어 있는 관계까지 함께 지우라는 뜻입니다(교안_01 마지막 절에 나옵니다).
run_cypher("MATCH (n) DETACH DELETE n")
# 확인: MATCH (n) RETURN n 은 남은 노드를 한 줄씩 돌려주므로 그 행 수가 곧 노드 개수다
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

초기화 완료. 남은 노드: 0


In [3]:
# [제공 코드] 스타트업 "노바랩스" 조직 그래프 적재: 이 셀은 실행만 하세요.
# 직원·팀·프로젝트 노드와 소속(WORKS_IN)·배정(ASSIGNED_TO)·소유(OWNS) 관계를 CREATE 로 만듭니다.
# 1) 넣을 값을 파이썬 리스트로 먼저 적어 둔다. years 는 뒤에서 조건 비교에 쓰인다
employees = [
    {"name": "김서준", "role": "백엔드", "years": 5, "team": "개발팀", "project": "결제시스템"},
    {"name": "정민재", "role": "백엔드", "years": 2, "team": "개발팀", "project": "결제시스템"},
    {"name": "박도윤", "role": "데이터", "years": 4, "team": "개발팀", "project": "결제시스템"},
    {"name": "이하은", "role": "프론트엔드", "years": 3, "team": "개발팀", "project": "앱개편"},
    {"name": "최지우", "role": "디자인", "years": 1, "team": "디자인팀", "project": "앱개편"},
]
projects = [
    {"name": "결제시스템", "status": "진행", "deadline": "2026-12-31"},
    {"name": "앱개편", "status": "완료", "deadline": "2026-03-31"},
]

team_projects = [("개발팀", "결제시스템"), ("디자인팀", "앱개편")]  # 팀이 맡은(소유한) 프로젝트

# 2) 프로젝트 노드부터 만든다. 마감일은 글자가 아니라 date() 로 만든 날짜 값으로 담는다
for p in projects:
    run_cypher(
        "CREATE (:Project {name: $name, status: $status, deadline: date($deadline)})",
        name=p["name"], status=p["status"], deadline=p["deadline"],
    )
# 3) 팀 노드. 팀은 이름뿐이라 속성이 하나다
for t in ["개발팀", "디자인팀"]:
    run_cypher("CREATE (:Team {name: $name})", name=t)
# 4) 팀 -> 프로젝트 관계(OWNS): 오늘 체인 패턴에서 두 번째 화살표로 쓰인다
for team, proj in team_projects:
    run_cypher(
        "MATCH (t:Team {name: $team}), (p:Project {name: $proj}) CREATE (t)-[:OWNS]->(p)",
        team=team, proj=proj,
    )
# 5) 직원 한 명마다 노드 1개 + 관계 2개를 만든다. 관계는 양쪽 노드가 이미 있어야 그을 수 있어 순서가 중요하다
for e in employees:
    run_cypher(
        "CREATE (:Employee {name: $name, role: $role, years: $years})",
        name=e["name"], role=e["role"], years=e["years"],
    )
    # 소속: 직원 -> 팀
    run_cypher(
        "MATCH (e:Employee {name: $name}), (t:Team {name: $team}) "
        "CREATE (e)-[:WORKS_IN]->(t)",
        name=e["name"], team=e["team"],
    )
    # 배정: 직원 -> 프로젝트
    run_cypher(
        "MATCH (e:Employee {name: $name}), (p:Project {name: $project}) "
        "CREATE (e)-[:ASSIGNED_TO]->(p)",
        name=e["name"], project=e["project"],
    )

print("적재한 노드 수:", len(run_cypher("MATCH (n) RETURN n")))   # 직원 5 + 팀 2 + 프로젝트 2

적재한 노드 수: 9


시연은 위의 **노바랩스 조직** 그래프로 하고, `🖐️ 함께 따라하기` 는 아래에서 적재하는 **캠퍼스라운지 동아리** 그래프로 합니다. 배운 문법을 **다른 데이터에 옮겨 써 보는 것**이 따라하기의 목적이라 데이터를 나눠 두었습니다. 두 그래프는 레이블이 달라 한 데이터베이스에 함께 있어도 서로 섞이지 않습니다.

| 노드 | 속성 | 관계 |
|---|---|---|
| `Student`(학생) | `name`, `major`(전공), `grade`(학년) | `BELONGS_TO`: 학생에서 동아리로 |
| `Club`(동아리) | `name` | `HOSTS`: 동아리에서 행사로 |
| `Event`(행사) | `name`, `status` | `JOINS`: 학생에서 행사로 |

아래 셀이 적재할 동아리 그래프의 전체 모습입니다. 학생 5명, 동아리 2개, 행사 3개가 세 종류의 관계로 이어져 있습니다. 조직 그래프와 **레이블만 다르고 짜임은 같습니다.**

<img src="images/club-graph-overview.png" width="820">

In [4]:
# [제공 코드] 따라하기용 "캠퍼스라운지" 동아리 그래프 적재: 이 셀은 실행만 하세요.
# 학생·동아리·행사 노드와 소속(BELONGS_TO)·주최(HOSTS)·참가(JOINS) 관계를 만듭니다.
# 위 조직 그래프와 만드는 순서가 같고, 레이블이 달라 한 DB 에 함께 있어도 섞이지 않습니다.
# 1) 넣을 값을 파이썬 리스트로 먼저 적어 둔다. grade(학년)는 뒤에서 조건 비교에 쓰인다
students = [
    {"name": "윤도현", "major": "경영",   "grade": 3, "club": "사진동아리", "event": "봄전시회"},
    {"name": "서지안", "major": "컴퓨터", "grade": 1, "club": "사진동아리", "event": "봄전시회"},
    {"name": "노태윤", "major": "디자인", "grade": 4, "club": "사진동아리", "event": "출사모임"},
    {"name": "임하늘", "major": "경영",   "grade": 2, "club": "밴드동아리", "event": "가을공연"},
    {"name": "구본재", "major": "컴퓨터", "grade": 3, "club": "밴드동아리", "event": "가을공연"},
]
events = [
    {"name": "봄전시회", "status": "모집중"},
    {"name": "출사모임", "status": "모집중"},
    {"name": "가을공연", "status": "마감"},
]

club_events = [("사진동아리", "봄전시회"), ("사진동아리", "출사모임"), ("밴드동아리", "가을공연")]

# 2) 행사 노드
for e in events:
    run_cypher("CREATE (:Event {name: $name, status: $status})", name=e["name"], status=e["status"])
# 3) 동아리 노드
for c in ["사진동아리", "밴드동아리"]:
    run_cypher("CREATE (:Club {name: $name})", name=c)
# 4) 동아리 -> 행사 관계(HOSTS): 사진동아리는 행사를 둘 여니 화살표가 두 개 나간다
for club, event in club_events:
    run_cypher(
        "MATCH (c:Club {name: $club}), (v:Event {name: $event}) CREATE (c)-[:HOSTS]->(v)",
        club=club, event=event,
    )
# 5) 학생 한 명마다 노드 1개 + 관계 2개(소속·참가)
for s in students:
    run_cypher(
        "CREATE (:Student {name: $name, major: $major, grade: $grade})",
        name=s["name"], major=s["major"], grade=s["grade"],
    )
    # 소속: 학생 -> 동아리
    run_cypher(
        "MATCH (s:Student {name: $name}), (c:Club {name: $club}) "
        "CREATE (s)-[:BELONGS_TO]->(c)",
        name=s["name"], club=s["club"],
    )
    # 참가: 학생 -> 행사
    run_cypher(
        "MATCH (s:Student {name: $name}), (v:Event {name: $event}) "
        "CREATE (s)-[:JOINS]->(v)",
        name=s["name"], event=s["event"],
    )

print("적재한 학생 수:", len(run_cypher("MATCH (s:Student) RETURN s")))   # Student 레이블만 세므로 조직 그래프는 안 잡힌다

적재한 학생 수: 5


---
# 1. 조건으로 거르기: WHERE

패턴만으로는 "근속 3년 **이상**" 같은 조건을 담을 수 없습니다. 그 자리를 맡는 것이 `WHERE` 입니다. 1-1 에서 **비교 연산자** 하나씩을, 1-2 에서 그 조건들을 **AND·OR 로 묶는 법**을, 1-3 에서 조건을 **뒤집는 법(NOT)** 과 값이 아예 비어 있는 노드를 다루는 법을 봅니다.

## 1-1. WHERE 와 비교 연산자

### 왜 필요할까요?
속성 map `{role: '백엔드'}` 는 **정확히 같은 값**만 찾습니다. "근속 3년 **이상**"이나 "근속 3년 **이하**" 같은 **비교**는 담을 수 없습니다. 이런 조건을 거는 자리가 **WHERE** 입니다.

### 문법: MATCH … WHERE … RETURN
```text
MATCH (e:Employee) WHERE e.years >= 3 RETURN e.name AS name
```

- `WHERE` 는 MATCH 로 찾은 것 중 **조건을 만족하는 것만** 남깁니다.
- 비교 연산자: `=`, `<>`(다름), `<`, `<=`, `>`, `>=`.

우리 조직 그래프의 직원에는 근속연수 `years` 속성이 있습니다. 이걸로 비교를 걸어 봅니다. 조건을 **여러 개 묶는 법**은 1-2 에서 이어서 다룹니다.

속성 map 과 무엇이 다른지 눈금 위에 놓고 보면 분명합니다.

<img src="images/where-vs-propmap.png" width="820">

In [5]:
# 근속 3년 이상인 직원: 속성 map 으로는 못 하는 '이상' 조건을 WHERE 로 건다
# 읽는 순서: MATCH 가 Employee 를 전부 모으고 -> WHERE 가 years >= 3 인 것만 남기고 -> RETURN 이 이름만 꺼낸다
rows = run_cypher("MATCH (e:Employee) WHERE e.years >= 3 RETURN e.name AS name")
print("근속 3년 이상:", sorted(r['name'] for r in rows))

근속 3년 이상: ['김서준', '박도윤', '이하은']


In [7]:
# 같지 않다(<>): 백엔드가 아닌 직원만 남긴다
# Cypher 의 '다르다' 는 파이썬의 != 가 아니라 <> 다
rows = run_cypher("MATCH (e:Employee) WHERE e.role <> '백엔드' RETURN e.name AS name")
print("백엔드가 아닌 직원:", sorted(r['name'] for r in rows))

백엔드가 아닌 직원: ['박도윤', '이하은', '최지우']


### 문법: 날짜도 크기를 비교한다
비교 연산자는 숫자에만 쓰는 것이 아닙니다. 교안_01 에서 본 대로 `date()` 로 적어 둔 날짜는 **앞뒤를 견줄 수 있는 값**이라 `>=`·`<` 가 그대로 통합니다. 시드의 프로젝트에는 마감일 `deadline` 이 날짜로 들어 있습니다.

In [8]:
# 마감일이 2026-07-01 이후인 프로젝트. 날짜끼리는 >= 로 어느 쪽이 뒤인지 견줄 수 있다
rows = run_cypher(
    "MATCH (p:Project) WHERE p.deadline >= date('2026-07-01') "
    "RETURN p.name AS name, p.deadline AS deadline"
)
print(rows)

[{'name': '결제시스템', 'deadline': neo4j.time.Date(2026, 12, 31)}]


이제 교안_01 에서 본 함정이 **조회에서 어떻게 드러나는지** 봅니다. 같은 마감일을 따옴표로만 적어 둔 프로젝트를 잠깐 넣고, 똑같은 조건을 걸어 봅니다.

In [9]:
# 마감일을 date() 가 아니라 따옴표로 적은 프로젝트를 하나 넣는다
run_cypher("CREATE (:Project {name: '글자마감', deadline: '2026-12-31'})")
hit = run_cypher(
    "MATCH (p:Project {name: '글자마감'}) WHERE p.deadline >= date('2026-07-01') "
    "RETURN p.name AS name"
)
run_cypher("MATCH (p:Project {name: '글자마감'}) DELETE p")   # 뒤 절에 섞이지 않게 지운다
print("글자로 적은 마감일이 조건에 걸렸나:", hit)

글자로 적은 마감일이 조건에 걸렸나: []


> **에러가 나지 않습니다.** 빈 결과가 조용히 돌아올 뿐이죠. 2026-12-31 은 분명 2026-07-01 보다 뒤인데도 걸리지 않았습니다. 글자와 날짜는 서로 견줄 수 있는 값이 아니라서, 1-3 에서 볼 **널**과 똑같이 조건에서 그냥 빠집니다.

> 이것이 적재할 때 날짜를 **날짜로** 넣어야 하는 이유입니다. 글자로 넣어 두면 조회가 틀렸다고 알려 주지 않고 **결과만 조용히 비어** 나옵니다. 눈치채기가 아주 어렵습니다.

### 🖐️ 함께 따라하기: 3학년 미만 학생 찾기

여기서부터는 **동아리 그래프**(캠퍼스라운지)로 옮겨 같은 문법을 써 봅니다. 학년(`grade`)이 **3보다 작은**(`<`) 학생을 찾으세요. `MATCH (s:Student)` 로 학생을 찾고 `WHERE` 에 비교 조건을 하나 겁니다. `s.name` 을 별칭 `name` 으로 RETURN 해 이름을 정렬해 출력하세요.

**확인 기준**: `['서지안', '임하늘']` **두 명**입니다(`<` 는 3학년을 포함하지 않습니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 Student 를 찾는다
# 2) WHERE 에 grade 가 3 보다 작은 조건을 건다(<= 가 아니라 <)
# 3) s.name 을 별칭 name 으로 RETURN 하고, 이름을 정렬해 출력한다

In [10]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 Student 를 찾는다
# 2) WHERE 에 grade 가 3 보다 작은 조건을 건다(<= 가 아니라 <)
# 3) s.name 을 별칭 name 으로 RETURN 하고, 이름을 정렬해 출력한다

rows = run_cypher("MATCH (s:Student) WHERE s.grade < 3 RETURN s")
print(rows)

[{'s': {'major': '컴퓨터', 'grade': 1, 'name': '서지안'}}, {'s': {'major': '경영', 'grade': 2, 'name': '임하늘'}}]


### ✅ 바로 확인 퀴즈

**1.** 속성 map `{years: 3}` 과 `WHERE e.years >= 3` 의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

속성 map 은 **정확히 3년**인 직원만, `WHERE e.years >= 3` 은 **3년 이상**(3, 4, 5, …)인 직원을 모두 찾습니다. 비교 조건은 WHERE 로만 걸 수 있습니다.

</details>

**2.** Cypher 에서 "같지 않다" 는 어떤 기호로 쓰나요?

<details><summary>정답 보기</summary>

**`<>`** 입니다. 파이썬의 `!=` 와 뜻은 같지만 기호가 다릅니다.

</details>

---
## 1-2. AND·OR 로 조합하기

### 왜 필요할까요?
조건이 하나로 끝나는 일은 드뭅니다. "아이스이면서 싼 것", "핫이거나 비싼 것" 처럼 둘을 묶어야 하죠. 그때 쓰는 것이 **`AND`**(둘 다)와 **`OR`**(하나라도)입니다.

### 문법: 조건 묶기
```text
WHERE e.role = '백엔드' AND e.years <= 3
WHERE e.role = '백엔드' OR  e.years <= 3
```

- **`AND`** 는 두 조건을 **모두** 만족해야 통과합니다. 그래서 결과가 좁아집니다.
- **`OR`** 는 **하나라도** 만족하면 통과합니다. 그래서 결과가 넓어집니다.
- 셋 이상을 섞을 때는 **괄호**로 묶어 순서를 분명히 합니다.

같은 두 조건이라도 무엇으로 묶느냐에 따라 결과가 이렇게 갈립니다.

<img src="images/and-or.png" width="820">

In [11]:
# 두 조건을 AND 로 조합: 백엔드이면서 근속 3년 이하(<= 는 3 년째인 사람도 포함한다)
rows = run_cypher(
    "MATCH (e:Employee) WHERE e.role = '백엔드' AND e.years <= 3 RETURN e.name AS name"
)
print("백엔드 & 3년 이하:", sorted(r['name'] for r in rows))   # 같은 백엔드라도 5년차 김서준은 빠진다

백엔드 & 3년 이하: ['정민재']


In [12]:
# 바로 위와 '같은 두 조건' 을 OR 로만 바꿔 잇는다. 그래야 넓어졌다는 말이 근거를 얻는다
rows = run_cypher(
    "MATCH (e:Employee) WHERE e.role = '백엔드' OR e.years <= 3 RETURN e.name AS name"
)
print("백엔드 | 3년 이하:", sorted(r['name'] for r in rows))   # AND 는 한 명이었는데 이쪽은 넷이다

백엔드 | 3년 이하: ['김서준', '이하은', '정민재', '최지우']


> `AND` 는 **둘 다** 만족해야, `OR` 는 **하나라도** 만족하면 통과입니다. 속성 map 과 WHERE 를 섞어도 됩니다: `MATCH (e:Employee {role: '백엔드'}) WHERE e.years >= 3` 도 같은 뜻입니다.

### 🖐️ 함께 따라하기: OR 로 조합

여기서부터는 **동아리 그래프**(캠퍼스라운지)로 옮겨 같은 문법을 써 봅니다. **전공이 `'컴퓨터'` 이거나 학년이 3 이상**인 학생을 찾으세요. `MATCH (s:Student)` 로 학생을 찾고, `WHERE` 에 두 조건을 **OR** 로 잇습니다. `s.name` 을 별칭 `name` 으로 RETURN 해 이름을 정렬해 출력하세요.

**확인 기준**: `['구본재', '노태윤', '서지안', '윤도현']` **네 명**입니다(구본재는 두 조건을 모두 만족하지만 한 번만 나옵니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 Student 를 찾는다
# 2) WHERE 에 major 가 '컴퓨터' 인 조건과 grade 가 3 이상인 조건을 OR 로 잇는다
# 3) s.name 을 별칭 name 으로 RETURN 하고, 이름을 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MATCH (e:Employee {role: '백엔드'}) WHERE e.years >= 3` 처럼 속성 map 과 WHERE 를 섞어 써도 되나요?

<details><summary>정답 보기</summary>

됩니다. 패턴 안의 속성 map 도 결국 하나의 조건이라, 두 조건을 `AND` 로 이은 것과 같은 뜻입니다. `MATCH (e:Employee) WHERE e.role = '백엔드' AND e.years >= 3` 과 결과가 같습니다.

</details>

**2.** `WHERE e.role = '백엔드' AND e.years <= 3` 과 `... OR ...` 는 결과가 어떻게 다른가요?

<details><summary>정답 보기</summary>

`AND` 는 **두 조건을 모두** 만족하는 직원만, `OR` 는 **둘 중 하나라도** 만족하는 직원을 찾습니다. 그래서 보통 `OR` 쪽이 더 많이 걸립니다.

</details>

---
## 1-3. NOT 과 IS NULL

### 왜 필요할까요?
"백엔드가 **아닌** 사람" 은 1-1 에서 `<>` 로 찾았습니다. 같은 뜻을 조건 앞에 **`NOT`** 을 붙여 쓸 수도 있습니다. 그런데 여기에 함정이 하나 숨어 있습니다. **속성이 아예 없는 노드**는 `= '백엔드'` 에도, `NOT = '백엔드'` 에도 걸리지 않습니다. 현실 데이터에는 값이 비어 있는 칸이 늘 있으므로, 그런 노드를 따로 찾는 문법(`IS NULL`)까지 알아야 셈이 맞습니다.

### 문법: NOT · IS NULL · IS NOT NULL
```text
MATCH (e:Employee) WHERE NOT e.role = '백엔드' RETURN e.name AS name
MATCH (e:Employee) WHERE e.role IS NULL        RETURN e.name AS name
MATCH (e:Employee) WHERE e.role IS NOT NULL    RETURN e.name AS name
```

- **`NOT`** 은 바로 뒤에 오는 조건을 통째로 뒤집습니다. `NOT e.role = '백엔드'` 는 `e.role <> '백엔드'` 와 **같은 뜻**입니다.
- 속성이 없는 노드에서 `e.role` 은 값이 아니라 **널(null)** 입니다. 널을 무엇과 비교해도 "맞다"도 "아니다"도 아니어서, 그 노드는 **양쪽 조건 모두에서 빠집니다**.
- 그래서 값이 **비어 있다는 것 자체**를 물을 때는 `= null` 이 아니라 **`IS NULL`** 을 씁니다. 반대로 값이 채워진 것만 남기려면 **`IS NOT NULL`** 입니다.

널이 낀 자리에서 셈이 어떻게 어긋나는지 그림으로 보면 이렇습니다.

<img src="images/not-isnull.png" width="820">

In [13]:
# NOT 은 뒤에 오는 조건을 통째로 뒤집는다. 1-1 에서 <> 로 얻은 결과와 같은지 눈으로 견준다
rows = run_cypher("MATCH (e:Employee) WHERE NOT e.role = '백엔드' RETURN e.name AS name")
print("NOT 으로 쓴 결과:", sorted(r['name'] for r in rows))   # 1-1 의 <> 결과와 같다

NOT 으로 쓴 결과: ['박도윤', '이하은', '최지우']


이제 함정을 만들 차례입니다. 아직 **역할이 정해지지 않은 신입** 한 명을 넣습니다. 빈 문자열을 넣는 것이 아니라 `role` 속성 자체를 **적지 않습니다**. 현실의 적재 데이터에서 값이 비는 칸이 바로 이 모양입니다.

In [14]:
# 이 절의 함정을 눈으로 보려면 '값이 비어 있는' 직원이 하나 필요하다
# role 을 아예 적지 않는다. 빈 문자열('')과 '속성이 없다'는 서로 다른 상태다
run_cypher("CREATE (:Employee {name: '한지민', years: 1})")
print("직원 수:", len(run_cypher("MATCH (e:Employee) RETURN e")))   # 시드 다섯 명에 한지민이 더해진다

직원 수: 6


In [15]:
# 서로 정반대인 두 조건을 각각 걸어 본다. 둘을 합치면 여섯 명이 다 나와야 할 것 같지만
baek = run_cypher("MATCH (e:Employee) WHERE e.role = '백엔드' RETURN e.name AS name")
not_baek = run_cypher("MATCH (e:Employee) WHERE NOT e.role = '백엔드' RETURN e.name AS name")
# role 이 없는 한지민은 '맞다' 쪽에도 '아니다' 쪽에도 들어가지 않아 한 명이 비는 것으로 나온다
print("백엔드 + 백엔드 아님:", len(baek) + len(not_baek))   # 직원 수보다 하나 모자란다. 그 한 명이 이 절의 주제다

백엔드 + 백엔드 아님: 5


In [16]:
# 빠진 한 명을 찾는 조건이 IS NULL 이다. = null 이라고 쓰지 않는다
empty = run_cypher("MATCH (e:Employee) WHERE e.role IS NULL RETURN e.name AS name")
filled = run_cypher("MATCH (e:Employee) WHERE e.role IS NOT NULL RETURN e.name AS name")
print("역할이 비어 있는 직원:", sorted(r['name'] for r in empty))
print("역할이 채워진 직원 수:", len(filled))

역할이 비어 있는 직원: ['한지민']
역할이 채워진 직원 수: 5


> 적재 상태를 점검할 때 `IS NULL` 은 아주 자주 씁니다. "이번에 넣은 노드 중 필수 속성이 비어 있는 것은 무엇인가" 를 묻는 것이 곧 이 조건입니다. 반대로 값이 채워진 것만 골라 계산하고 싶을 때는 `IS NOT NULL` 로 먼저 걸러 두면 셈이 어긋나지 않습니다.

### 🖐️ 함께 따라하기: 전공이 비어 있는 학생 찾기

다시 **동아리 그래프**입니다. 아직 전공을 정하지 않은 신입생 **한소민**(1학년)을 `major` 속성 **없이** `CREATE` 로 넣으세요. 그런 다음 두 가지를 확인합니다.

1. `WHERE NOT s.major = '컴퓨터'` 로 찾은 학생 이름을 정렬해 출력한다.
2. `WHERE s.major IS NULL` 로 찾은 학생 이름을 정렬해 출력한다.

**확인 기준**: 1번은 `['노태윤', '윤도현', '임하늘']` 세 명이고 **한소민은 거기 없습니다**. 2번이 `['한소민']` 을 돌려줍니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE 로 한소민(grade 1)을 넣는다. major 는 아예 적지 않는다
# 2) WHERE NOT s.major = '컴퓨터' 로 찾은 이름을 정렬해 출력한다(한소민은 안 나온다)
# 3) WHERE s.major IS NULL 로 찾은 이름을 정렬해 출력한다(한소민만 나온다)

In [17]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE 로 한소민(grade 1)을 넣는다. major 는 아예 적지 않는다
# 2) WHERE NOT s.major = '컴퓨터' 로 찾은 이름을 정렬해 출력한다(한소민은 안 나온다)
# 3) WHERE s.major IS NULL 로 찾은 이름을 정렬해 출력한다(한소민만 나온다)

run_cypher("CREATE (s:Student {name:'한소민', grade:1})")
print(run_cypher("MATCH (s:Student) WHERE NOT s.major = '컴퓨터' RETURN s"))
print(run_cypher("MATCH (s:Student) WHERE s.major IS NULL RETURN s"))

[{'s': {'major': '경영', 'grade': 3, 'name': '윤도현'}}, {'s': {'major': '디자인', 'grade': 4, 'name': '노태윤'}}, {'s': {'major': '경영', 'grade': 2, 'name': '임하늘'}}]
[{'s': {'grade': 1, 'name': '한소민'}}]


### ✅ 바로 확인 퀴즈

**1.** `WHERE NOT e.role = '백엔드'` 와 `WHERE e.role <> '백엔드'` 는 결과가 다른가요?

<details><summary>정답 보기</summary>

**같습니다.** `NOT` 은 뒤의 조건을 뒤집는 것이고, `<>` 는 "같지 않다" 를 한 기호로 쓴 것이라 뜻이 같습니다. 조건이 길어지면 `NOT (…)` 쪽이 읽기 편할 때가 있습니다.

</details>

**2.** `role` 속성이 아예 없는 직원은 왜 `= '백엔드'` 에도 `NOT = '백엔드'` 에도 나오지 않나요? 그 직원을 찾으려면 어떤 조건을 써야 하나요?

<details><summary>정답 보기</summary>

속성이 없으면 그 값은 **널**이고, 널을 무엇과 비교해도 참도 거짓도 되지 않아 양쪽 조건에서 모두 빠집니다. 그런 노드는 **`WHERE e.role IS NULL`** 로 찾습니다(`= null` 은 쓰지 않습니다).

</details>

**3.** "역할이 채워져 있으면서 근속 3년 이상" 인 직원은 어떻게 쓰나요?

<details><summary>정답 보기</summary>

**`WHERE e.role IS NOT NULL AND e.years >= 3`** 입니다. `IS NOT NULL` 도 다른 조건과 똑같이 `AND`·`OR` 로 이어 쓸 수 있습니다.

</details>

---
# 2. 여러 노드를 잇는 패턴

노드 둘을 잇는 것까지는 지난 시간에 했습니다. 이제 셋 이상을 한 문장에 잇습니다. 2-1 에서 거쳐 가는 모양(체인·역방향)을, 2-2 에서 가운데를 함께 가리키는 모양(공유)을 봅니다.

## 2-1. 체인과 역방향

### 왜 필요할까요?
"내 팀이 맡은 프로젝트는?" 같은 질문은 노드 **셋**을 거칩니다. 이렇게 패턴을 길게 이으면 한 문장으로 **연결을 따라가는** 쿼리가 됩니다. 이 조직 그래프에는 팀이 프로젝트를 맡는 `OWNS` 관계(`팀 → 프로젝트`)도 있습니다.

### 문법 1) 체인: 화살표를 같은 방향으로 이어 그리기
```text
(e)-[:WORKS_IN]->(t)-[:OWNS]->(p)
```

- 직원 `e` → 소속 팀 `t` → 그 팀이 맡은 프로젝트 `p` 로 **한 방향으로** 두 관계를 잇습니다.
- 가운데 팀 `t` 는 **거쳐가는** 노드입니다. 두 관계를 연결하는 다리 역할이죠.
- 이렇게 **서로 다른 관계**(`WORKS_IN`, `OWNS`)를 이어 "내 팀이 맡은 프로젝트"까지 한 번에 갑니다.

In [18]:
# 김서준이 속한 팀이 맡은 프로젝트 찾기: 직원 -> 팀 -> 프로젝트 체인
# 화살표 두 개를 한 줄로 이었다. 가운데 t 는 거쳐 가기만 하는 다리라 RETURN 에 넣지 않는다
rows = run_cypher(
    "MATCH (e:Employee {name: '김서준'})-[:WORKS_IN]->(t)-[:OWNS]->(p) RETURN p.name AS name"
)
print("김서준 팀이 맡은 프로젝트:", sorted(r['name'] for r in rows))

김서준 팀이 맡은 프로젝트: ['결제시스템']


### 문법 2) 역방향: 화살표를 거꾸로 읽기
```text
(p)<-[:ASSIGNED_TO]-(e)
```

- 지금까지는 관계를 **왼쪽에서 오른쪽으로**(`->`) 그렸습니다. 하지만 질문의 출발점이 반대쪽일 때가 많습니다. "이 프로젝트에 **배정된 사람**은?" 처럼요.
- 이때는 화살표를 **거꾸로**(`<-`) 그려 프로젝트에서 직원 쪽으로 거슬러 갑니다.
- **화살표를 거꾸로 읽어도 관계의 종류와 방향은 그대로**입니다. `ASSIGNED_TO` 는 여전히 직원에서 프로젝트로 향합니다. 바뀌는 건 우리가 **어디서 출발해 읽느냐**뿐입니다.
- 역방향도 이어서 길게 쓸 수 있습니다. `(p)<-[:OWNS]-(t)<-[:WORKS_IN]-(e)` 는 프로젝트에서 출발해 그 프로젝트를 맡은 팀으로, 다시 그 팀에 속한 직원으로 두 단계를 거슬러 올라갑니다.

두 문장이 가리키는 화살표는 똑같습니다. 바뀌는 것은 읽는 출발점뿐입니다.

<img src="images/chain-reverse.png" width="820">

In [19]:
# 결제시스템에 배정된 직원 찾기: 프로젝트에서 직원 쪽으로 화살표를 거슬러 읽는다
# <-[:ASSIGNED_TO]- 로 그려도 관계 자체는 여전히 직원 -> 프로젝트 방향이다. 바뀐 건 출발점뿐
rows = run_cypher(
    "MATCH (p:Project {name: '결제시스템'})<-[:ASSIGNED_TO]-(e:Employee) RETURN e.name AS name"
)
print("결제시스템 배정 직원:", sorted(r['name'] for r in rows))

결제시스템 배정 직원: ['김서준', '박도윤', '정민재']


> 거슬러 올라가는 칸을 **연달아 두 번** 쓸 수도 있습니다. 아래는 프로젝트에서 출발해 그 프로젝트를 맡은 **팀**으로, 다시 그 팀에 속한 **직원**으로 두 칸을 거슬러 올라갑니다. 화살표가 둘 다 왼쪽을 향하고, 앞 칸의 도착점(팀)이 다음 칸의 출발점이 됩니다.

In [20]:
# 두 칸 연속으로 거슬러 오르기: 결제시스템 -> 그 프로젝트를 맡은 팀 -> 그 팀에 속한 직원
# 화살표가 둘 다 <- 다. 앞 칸이 잡은 팀 t 를 다음 칸이 그대로 이어받는다
rows = run_cypher(
    "MATCH (p:Project {name: '결제시스템'})<-[:OWNS]-(t:Team)<-[:WORKS_IN]-(e:Employee) "
    "RETURN e.name AS name"
)
# 결과 순서는 정해져 있지 않으니 여기서도 파이썬에서 정렬한다
print("결제시스템을 맡은 팀의 직원:", sorted(r['name'] for r in rows))   # 배정 여부와는 다른 질문이다

결제시스템을 맡은 팀의 직원: ['김서준', '박도윤', '이하은', '정민재']


> 바로 위 "결제시스템에 **배정된** 직원"(세 명)과 견줘 보세요. 이번 답은 **네 명**입니다. `ASSIGNED_TO` 는 사람이 그 프로젝트에 직접 배정됐다는 사실이고, `OWNS`+`WORKS_IN` 두 칸은 그 프로젝트를 맡은 팀에 속해 있다는 사실입니다. **어느 관계를 몇 칸 따라가느냐가 곧 질문**입니다.

> 가운데 노드에서 **양쪽으로 갈라지는** 모양도 자주 씁니다: 한쪽은 거슬러(`<-`), 다른 쪽은 나아가며(`->`) 두 조건을 동시에 겁니다. 아래는 "개발팀 소속이면서 앱개편에 배정된 직원" 입니다.

In [21]:
# 가운데 직원 e 에서 양쪽으로 갈라지는 패턴: 개발팀 소속이면서 앱개편에 배정된 직원
# 왼쪽은 거슬러(<-), 오른쪽은 나아가며(->) 두 조건을 한 문장에 함께 건다
rows = run_cypher(
    "MATCH (:Team {name: '개발팀'})<-[:WORKS_IN]-(e:Employee)-[:ASSIGNED_TO]->(:Project {name: '앱개편'}) "
    "RETURN e.name AS name"
)
print("개발팀 & 앱개편:", sorted(r['name'] for r in rows))   # 개발팀 4명 중 앱개편을 맡은 한 명

개발팀 & 앱개편: ['이하은']


### 🖐️ 함께 따라하기: 동아리를 거쳐 행사까지

다시 **동아리 그래프**입니다. **윤도현**이 속한 동아리가 **여는 행사**를 찾으세요. 학생에서 `BELONGS_TO` 로 동아리에, 동아리에서 `HOSTS` 로 행사에 이어지는 **체인 패턴**을 그리면 됩니다. 행사 이름을 별칭 `name` 으로 RETURN 해 정렬 출력하세요.

**확인 기준**: `['봄전시회', '출사모임']` **두 개**입니다. 시연의 김서준은 프로젝트가 하나였는데 여기서 둘이 나오는 것은, 윤도현이 속한 사진동아리가 행사를 둘 열기 때문입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 윤도현에서 BELONGS_TO 로 동아리에, 다시 HOSTS 로 행사에 이어지는 패턴을 MATCH 한다
# 2) 행사 이름을 별칭 name 으로 RETURN 한다
# 3) 이름을 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 체인 패턴 `(e)-[:WORKS_IN]->(t)-[:OWNS]->(p)` 에서 가운데 `t`(팀)의 역할은 무엇인가요?

<details><summary>정답 보기</summary>

두 관계(`WORKS_IN`, `OWNS`)를 잇는 **다리(거쳐가는 노드)**입니다. 직원에서 팀을 거쳐 그 팀이 맡은 프로젝트까지 한 번에 이어집니다.

</details>

**2.** `(e)-[:ASSIGNED_TO]->(p)` 와 `(p)<-[:ASSIGNED_TO]-(e)` 는 **서로 다른 관계**를 가리키나요?

<details><summary>정답 보기</summary>

아닙니다. **같은 관계**입니다. `ASSIGNED_TO` 의 방향도 여전히 직원에서 프로젝트로 그대로입니다. 달라진 것은 우리가 **어느 쪽에서 출발해 읽느냐**뿐입니다.

</details>

---
## 2-2. 공유 패턴: 가운데 노드를 함께 가리키기

### 왜 필요할까요?
"같은 프로젝트에 배정된 동료는?" 은 앞 절의 체인과 모양이 다릅니다. 거쳐 가는 것이 아니라 **두 사람이 가운데 노드 하나를 함께 가리키는** 모양이기 때문입니다.

### 문법: 공유 패턴
```text
(a)-[:ASSIGNED_TO]->(p)<-[:ASSIGNED_TO]-(b)
```

- `a` 와 `b` 는 **같은 프로젝트 `p`** 에 배정된 두 직원입니다.
- 화살표 방향에 주의하세요: `a` 는 `p` 로 **나가고**(`->`), `b` 는 `p` 로 들어옵니다(`<-`). 둘 다 "프로젝트에 배정된다"를 뜻합니다.
- 체인이 **거쳐가는** 모양이라면, 이건 **가운데 노드를 공유**하는 모양입니다.

In [22]:
# 김서준과 같은 프로젝트에 배정된 동료 찾기: 가운데 프로젝트 p 를 공유하는 패턴
# a 와 b 가 같은 p 를 마주 본다. 한 문장에서 같은 화살표를 두 번 쓸 수 없어 김서준 자신은 빠진다
rows = run_cypher(
    "MATCH (a:Employee {name: '김서준'})-[:ASSIGNED_TO]->(p)<-[:ASSIGNED_TO]-(b:Employee) "
    "RETURN b.name AS name"
)
print("김서준의 프로젝트 동료:", sorted(r['name'] for r in rows))   # 결제시스템을 함께 맡은 둘

김서준의 프로젝트 동료: ['박도윤', '정민재']


두 모양을 나란히 두면 차이가 분명해집니다. 한쪽은 거쳐 가고, 다른 쪽은 가운데를 함께 가리킵니다.

<img src="images/chain-vs-shared.png" width="820">

> 공유 패턴의 결과에 **김서준 자신은 빠집니다**. 한 번 지나간 **화살표(관계)** 는 같은 `MATCH` 절 안에서 다시 쓸 수 없기 때문입니다. 김서준이 결제시스템으로 가는 화살표는 하나뿐인데, 자기 자신이 동료로 걸리려면 그 하나를 왼쪽과 오른쪽에 동시에 써야 합니다. 그래서 자기 자신은 빠집니다.

"같은 문장" 이 아니라 **"같은 `MATCH` 절"** 인 것이 중요합니다. 한 문장이라도 `MATCH` 를 **둘로 나눠 적으면** 이 규칙이 절마다 따로 적용돼 결과가 달라집니다. 아래에서 직접 봅니다.

In [23]:
# 같은 뜻으로 읽히지만 결과가 다르다. 위는 화살표 둘을 한 MATCH 절에 그렸고
# 아래는 같은 그림을 MATCH 두 절로 쪼갰다. 절이 다르면 같은 화살표를 다시 써도 된다
one = run_cypher(
    "MATCH (a:Employee {name: '김서준'})-[:ASSIGNED_TO]->(p)<-[:ASSIGNED_TO]-(b:Employee) "
    "RETURN b.name AS name"
)
two = run_cypher(
    "MATCH (a:Employee {name: '김서준'})-[:ASSIGNED_TO]->(p) "
    "MATCH (p)<-[:ASSIGNED_TO]-(b:Employee) "
    "RETURN b.name AS name"
)
print("한 절로 :", sorted(r['name'] for r in one))   # 자기 자신이 빠졌다
print("두 절로 :", sorted(r['name'] for r in two))   # 자기 자신이 남았다

한 절로 : ['박도윤', '정민재']
두 절로 : ['김서준', '박도윤', '정민재']


> 그래서 "동료" 를 찾을 때는 **화살표 둘을 한 `MATCH` 절에** 그려야 합니다. 나눠 적으면 자기 자신이 동료로 끼어듭니다. 뜻이 같아 보이는 두 문장이 이렇게 갈리는 자리라 기억해 둘 만합니다.

### 🖐️ 함께 따라하기: 같은 동아리 학생

다시 **동아리 그래프**입니다. **임하늘**과 같은 **동아리**에 속한 다른 학생을 찾으세요. 두 학생이 각각 `BELONGS_TO` 로 **같은 동아리 노드를 향하도록**(화살표가 마주 보도록) 이으면 됩니다. 학생의 이름을 별칭 `name` 으로 RETURN 해 정렬 출력하세요.

**확인 기준**: 임하늘과 같은 밴드동아리인 `['구본재']` **한 명**입니다(임하늘 자신은 빠집니다).

In [36]:
one = run_cypher("""
     MATCH (s:Student {name: '임하늘'})-[:BELONGS_TO]->(c:Club)<-[:BELONGS_TO]-(b:Student)
     RETURN b.name AS name
     """
)

print(one)

[{'name': '구본재'}]


In [37]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 임하늘에서 소속 동아리로 가고, 그 동아리로 다른 학생이 들어오게 잇는 패턴을 MATCH 한다
# 2) b.name 을 별칭 name 으로 RETURN 한다
# 3) 이름을 정렬해 출력한다

one = run_cypher("""
     MATCH (s:Student {name: '임하늘'})-[:BELONGS_TO]->(c:Club)<-[:BELONGS_TO]-(b:Student)
     RETURN b.name AS name
     """
)


two = run_cypher(
    "MATCH (s:Student {name: '임하늘'})-[:BELONGS_TO]->(c:Club) "
    "MATCH (c)<-[:BELONGS_TO]-(b:Student) "
    "RETURN b.name AS name"
)

print(one)
print(two)

[{'name': '구본재'}]
[{'name': '임하늘'}, {'name': '구본재'}]


### ✅ 바로 확인 퀴즈

**1.** 공유 패턴 `(a)-[:WORKS_IN]->(t)<-[:WORKS_IN]-(b)` 에서 두 화살표의 방향이 서로 반대인 이유는?

<details><summary>정답 보기</summary>

`WORKS_IN` 은 **직원에서 팀으로** 향하므로, `a`·`b` 두 직원이 각각 같은 팀 `t` 로 들어가려면 둘 다 `t` 를 향해야 합니다. 그래서 `t` 를 기준으로 화살표가 마주 봅니다.

</details>

**2.** 공유 패턴으로 "김서준의 동료"를 찾으면 왜 김서준 자신은 결과에 나오지 않나요?

<details><summary>정답 보기</summary>

한 번 지나간 **화살표(관계)** 를 같은 `MATCH` 절 안에서 다시 쓸 수 없기 때문입니다. 김서준이 결제시스템으로 가는 화살표는 하나뿐인데, 자기 자신이 동료로 걸리려면 그 하나를 왼쪽과 오른쪽에 동시에 써야 합니다.

</details>

**3.** 같은 그림을 `MATCH` **두 절로 쪼개** 적으면(`MATCH (a)-[:R]->(p)` 다음에 `MATCH (p)<-[:R]-(b)`) 김서준은 결과에 나올까요?

<details><summary>정답 보기</summary>

**나옵니다.** 화살표를 다시 쓸 수 없다는 규칙은 **`MATCH` 절 하나 안에서만** 적용되기 때문입니다. 절을 나누면 두 절이 같은 화살표를 각각 쓸 수 있어 김서준이 자기 동료로 걸립니다. 동료를 찾을 때는 **한 절에** 그려야 합니다.

</details>

---
# 3. 결과 다듬기

**찾는 것**과 **보여 주는 것**은 다른 일입니다. MATCH 는 패턴이 맞은 경우를 하나하나 행으로 돌려주므로 같은 값이 여러 번 나오고, 그 순서도 정해져 있지 않습니다. 3-1 에서 **행 수가 어떻게 정해지는지**를 먼저 이해하고, 3-2 에서 되풀이를 **`DISTINCT`** 로 접고, 3-3 에서 **`ORDER BY`·`LIMIT`** 으로 줄을 세워 위에서 몇 개만 받아 옵니다.

## 3-1. 결과는 몇 줄인가

### 왜 필요할까요?
패턴을 넓히면 같은 이름이 여러 번 나와 처음에는 버그처럼 보입니다. 버그가 아니라 규칙입니다. 이 규칙을 모르면 뒤에서 개수를 셀 때마다 틀립니다.

### 규칙: 행 수는 어떻게 정해지나
지금까지 데모는 노드 하나가 한 행이어서 행 수와 노드 수가 같아 보였습니다. 관계를 따라가는 패턴에서는 곧바로 달라집니다. **MATCH 는 패턴이 맞아떨어지는 경우를 전부 찾아, 그 하나하나를 한 행으로 돌려줍니다.** 노드를 세는 것이 아니라 **길을 세는 것**이죠.

그래서 같은 이름이 여러 번 나올 수 있습니다. 결제시스템에 세 명이 배정돼 있으면 "프로젝트 이름"을 물어도 결제시스템이 **세 번** 나옵니다. 배정 화살표가 셋이니 길도 셋이기 때문입니다. 처음 보면 버그 같지만 정상입니다.

In [39]:
# 배정 관계를 따라가되 프로젝트 이름만 돌려받는다
# 화살표 하나가 한 행이 되므로, 배정된 사람 수만큼 같은 프로젝트 이름이 되풀이된다
rows = run_cypher("MATCH (p:Project)<-[:ASSIGNED_TO]-(e:Employee) RETURN p.name AS name")
print("행 수:", len(rows))   # 프로젝트가 둘인데도 5. 배정 관계가 다섯 개라서다
print("돌려받은 이름:", sorted(r['name'] for r in rows))

행 수: 5
돌려받은 이름: ['결제시스템', '결제시스템', '결제시스템', '앱개편', '앱개편']


In [38]:
# 서로 다른 이름만 보고 싶으면 파이썬 집합(set)으로 중복을 없앤다
# {r['name'] for r in rows} 는 리스트가 아니라 집합을 만드는 컴프리헨션이다(중괄호)
print("서로 다른 프로젝트:", sorted({r['name'] for r in rows}))

서로 다른 프로젝트: ['박도윤', '정민재']


> 지금은 다섯 행을 **전부 받아 온 뒤** 파이썬에서 접었습니다. 같은 일을 **Cypher 문장 안에서** 하는 문법이 바로 다음 절의 `DISTINCT` 입니다. 둘 다 결과는 같지만, 접는 자리가 다릅니다.

그림으로 보면 이렇습니다. 노드가 아니라 화살표를 센다는 것이 핵심입니다.

<img src="images/pattern-rows.png" width="820">

### 규칙: 패턴을 나란히 적으면 행 수가 곱이 된다
행이 늘어나는 길은 하나 더 있습니다. 교안_01 의 1-4 에서 `MATCH (e), (t)` 처럼 **쉼표로 패턴 둘을 나란히** 적었죠. 그때는 양쪽 다 속성 map 으로 한 명씩 콕 집어 `1 x 1 = 1` 행이라 티가 나지 않았습니다. 범위를 넓히면 곧바로 드러납니다.

쉼표는 "둘을 각각 찾아라" 입니다. 두 패턴이 **서로 아무 변수도 공유하지 않으면**, 찾은 것끼리 **모든 짝**이 만들어집니다. 화살표를 세는 것이 아니라 **짝을 세는 것**이죠.

In [40]:
# 쉼표로 나란히 적은 두 패턴은 서로 무관하다. 그래서 팀 하나하나에 프로젝트 하나하나가 다 짝지어진다
teams = len(run_cypher("MATCH (t:Team) RETURN t"))
projects = len(run_cypher("MATCH (p:Project) RETURN p"))
pairs = run_cypher("MATCH (t:Team), (p:Project) RETURN t.name AS team, p.name AS project")
# 팀이 그 프로젝트를 맡았는지는 보지도 않는다. 둘을 이어 주는 화살표를 적지 않았기 때문이다
print(f"팀 {teams}개 · 프로젝트 {projects}개 -> 행 수 {len(pairs)}")

팀 2개 · 프로젝트 2개 -> 행 수 4


In [41]:
print(pairs)

[{'team': '개발팀', 'project': '결제시스템'}, {'team': '개발팀', 'project': '앱개편'}, {'team': '디자인팀', 'project': '결제시스템'}, {'team': '디자인팀', 'project': '앱개편'}]


> 팀이 그 프로젝트를 **맡았는지 아닌지는 보지도 않았습니다.** 둘을 이어 주는 화살표를 적지 않았으니까요. 관계로 이으려면 `MATCH (t:Team)-[:OWNS]->(p:Project)` 처럼 **한 패턴으로** 그려야 합니다. 쉼표는 이을 때 쓰는 것이 아니라 **따로 찾을 때** 쓰는 것입니다.

> `MATCH (a), (b)` 와 `MATCH (a) MATCH (b)` 는 **이렇게 변수를 공유하지 않을 때는 결과가 같습니다.** 둘 다 곱이 되죠. 갈리는 것은 변수를 공유할 때이고, 그게 2-2 에서 본 화살표 재사용 규칙입니다.

### 🖐️ 함께 따라하기: 동아리 이름이 몇 번 나오나

다시 **동아리 그래프**입니다. 학생이 `BELONGS_TO` 로 동아리에 이어지는 패턴에서 **동아리 이름**을 별칭 `name` 으로 RETURN 하세요. 먼저 **행 수**를 출력하고, 이어서 파이썬 **집합**으로 중복을 없앤 **서로 다른 동아리 이름**을 정렬해 출력하세요.

**확인 기준**: 행 수는 **5**(소속 화살표 다섯 개)이고, 서로 다른 이름은 `['밴드동아리', '사진동아리']` **둘**입니다(한 동아리에 여러 학생이 속해 있으니까요).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 학생에서 BELONGS_TO 로 동아리에 이어지는 패턴을 MATCH 하고 동아리 이름을 별칭 name 으로 RETURN 한다
# 2) 행 수를 출력한다
# 3) 집합 컴프리헨션으로 중복을 없앤 이름을 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MATCH (t:Team)<-[:WORKS_IN]-(e:Employee) RETURN t.name AS name` 을 실행하면 팀이 둘뿐인데도 행이 다섯 개 나옵니다. 왜 그럴까요?

<details><summary>정답 보기</summary>

MATCH 가 세는 것은 **노드가 아니라 패턴이 맞은 경우**이기 때문입니다. 소속 화살표가 다섯 개이니 길도 다섯 개이고, 그 하나하나가 한 행이 됩니다. 그래서 개발팀은 소속 직원 수만큼 되풀이됩니다.

</details>

**2.** 1-3 에서 넣은 신입 **한지민**은 아직 어느 팀에도 속해 있지 않습니다. 위 쿼리 결과에 한지민이 만든 행이 있을까요?

<details><summary>정답 보기</summary>

**없습니다.** 한지민에게는 `WORKS_IN` 화살표가 하나도 없어 패턴이 맞아떨어지는 경우가 생기지 않습니다. 행은 노드 하나마다가 아니라 **패턴이 맞은 길마다** 만들어지기 때문입니다.

</details>

---
## 3-2. DISTINCT 로 되풀이 접기

### 왜 필요할까요?
3-1 에서 우리는 다섯 행을 **전부 받아 온 뒤** 파이썬 집합으로 중복을 없앴습니다. 행이 다섯 개일 때는 괜찮지만, 수만 행이면 필요 없는 데이터를 통째로 끌어오는 셈입니다. 같은 일을 **DB 쪽에서** 먼저 해 주는 것이 `DISTINCT` 입니다.

### 문법: RETURN DISTINCT
```text
MATCH (p:Project)<-[:ASSIGNED_TO]-(e:Employee) RETURN DISTINCT p.name AS name
```

- `DISTINCT` 는 **`RETURN` 바로 뒤에 한 번만** 씁니다. `RETURN p.name DISTINCT` 처럼 뒤에 붙이지 않습니다.
- 열 하나에 붙는 것이 아니라 **RETURN 이 돌려주는 행 전체**가 같은지를 봅니다.
- 그래서 `RETURN DISTINCT p.name, e.name` 은 **두 값의 쌍**을 견줍니다. 쌍이 하나라도 다르면 다른 행이라 접히지 않습니다. "DISTINCT 를 썼는데 중복이 그대로다" 의 대부분이 이 경우입니다.

무엇을 기준으로 접는지 그림으로 보면 분명합니다.

<img src="images/distinct.png" width="820">

In [42]:
# 3-1 과 같은 패턴에 DISTINCT 만 얹었다. 접는 일을 파이썬이 아니라 DB 가 한다
rows = run_cypher("MATCH (p:Project)<-[:ASSIGNED_TO]-(e:Employee) RETURN DISTINCT p.name AS name")
print("행 수:", len(rows))   # 다섯 행이 아니라 접힌 두 행만 건너온다
print("서로 다른 프로젝트:", sorted(r['name'] for r in rows))

행 수: 2
서로 다른 프로젝트: ['결제시스템', '앱개편']


In [43]:
# 함정: DISTINCT 는 '행 전체'가 같은지를 본다. 사람 이름을 함께 돌려받으면
# 프로젝트 이름이 같아도 (프로젝트, 사람) 쌍이 달라 한 줄도 접히지 않는다
rows = run_cypher(
    "MATCH (p:Project)<-[:ASSIGNED_TO]-(e:Employee) "
    "RETURN DISTINCT p.name AS name, e.name AS who"
)
print("행 수:", len(rows))   # DISTINCT 를 썼는데도 3-1 과 같은 다섯 행이다

행 수: 5


> 파이썬 집합과 `DISTINCT` 는 **결과가 같고 접는 자리가 다릅니다.** 집합은 다 받아 온 뒤 파이썬에서, `DISTINCT` 는 받아 오기 전에 DB 에서 접습니다. 오가는 행 수를 줄이는 쪽이 대개 낫지만, 이미 받아 둔 결과를 파이썬에서 한 번 더 정리하는 것도 틀린 방법이 아닙니다.

### 🖐️ 함께 따라하기: DISTINCT 로 동아리 이름 접기

다시 **동아리 그래프**입니다. 3-1 따라하기에서 **파이썬 집합**으로 접었던 그 결과를 이번에는 **`RETURN DISTINCT`** 로 접어 보세요. 같은 `BELONGS_TO` 패턴을 쓰되 `RETURN DISTINCT c.name AS name` 으로 바꾸고, **행 수**와 **정렬한 이름 목록**을 출력합니다.

**확인 기준**: 행 수가 5 가 아니라 **2** 이고, 이름은 `['밴드동아리', '사진동아리']` 입니다.

In [46]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 3-1 과 같은 BELONGS_TO 패턴을 MATCH 한다
# 2) RETURN DISTINCT c.name AS name 으로 중복을 DB 에서 접는다
# 3) 행 수와 정렬한 이름 목록을 출력한다

print(run_cypher(
    "MATCH (c:Club)"
    "RETURN DISTINCT c.name AS name"
))
print("행 수:", len(rows))

[{'name': '사진동아리'}, {'name': '밴드동아리'}]
행 수: 2


### ✅ 바로 확인 퀴즈

**1.** `RETURN DISTINCT p.name, e.name` 은 왜 프로젝트 이름의 중복을 접어 주지 않나요?

<details><summary>정답 보기</summary>

`DISTINCT` 는 열 하나가 아니라 **행 전체**가 같은지를 보기 때문입니다. 프로젝트 이름이 같아도 사람 이름이 다르면 서로 다른 행이라 접히지 않습니다. 접고 싶은 값만 RETURN 에 남겨야 합니다.

</details>

**2.** 파이썬 집합으로 접는 것과 `DISTINCT` 로 접는 것은 무엇이 다른가요?

<details><summary>정답 보기</summary>

**접는 자리**가 다릅니다. 파이썬 집합은 모든 행을 **받아 온 뒤** 접고, `DISTINCT` 는 DB 가 접어 **줄어든 행만** 보냅니다. 결과는 같지만 오가는 데이터 양이 다릅니다.

</details>

---
## 3-3. ORDER BY 와 LIMIT

### 왜 필요할까요?
지금까지는 결과를 받아 온 뒤 파이썬 `sorted()` 로 정렬했습니다. 이름을 견주는 데는 그것으로 충분합니다. 하지만 **"근속이 가장 긴 세 명"** 처럼 **순서 자체가 답**일 때는 이야기가 다릅니다. 전부 받아 와서 자르는 대신, DB 에서 줄을 세우고 **위에서 세 줄만** 받아 오면 됩니다.

### 문법: ORDER BY … LIMIT
```text
MATCH (e:Employee) RETURN e.name AS name, e.years AS years
ORDER BY e.years DESC
LIMIT 3
```

- **`ORDER BY`** 는 기준 값으로 줄을 세웁니다. 기본은 **오름차순**이고, 뒤에 **`DESC`** 를 붙이면 내림차순입니다.
- **`LIMIT n`** 은 줄을 세운 뒤 **위에서 n 줄만** 남깁니다. `ORDER BY` 없이 `LIMIT` 만 쓰면 "상위 n" 이 아니라 **아무 n 줄**이 옵니다. 둘은 짝으로 씁니다.
- 결과를 받은 뒤 파이썬에서 다시 정렬하면 **애써 세운 줄이 흐트러집니다.** 이 절에서는 `sorted()` 를 쓰지 않고 받은 순서 그대로 찍습니다.
- **동점**이 있으면 그들끼리의 앞뒤는 보장되지 않습니다. `ORDER BY e.years, e.name` 처럼 **보조 정렬키**를 하나 더 두면 언제 실행해도 같은 순서가 나옵니다.

줄을 세우고 위에서 자르는 순서를 그림으로 보면 이렇습니다.

<img src="images/order-limit.png" width="820">

In [ ]:
# 근속이 가장 긴 세 명. DESC 로 내림차순 정렬한 뒤 LIMIT 3 으로 위에서 세 줄만 받는다
rows = run_cypher(
    "MATCH (e:Employee) RETURN e.name AS name, e.years AS years "
    "ORDER BY e.years DESC LIMIT 3"
)
# 돌려받은 순서가 곧 답이라 여기서는 sorted() 로 다시 흩뜨리지 않는다
for r in rows:
    print(r['name'], r['years'])

한지민 1
최지우 1
정민재 2


이번에는 반대쪽 끝, **근속이 짧은 순**으로 세 명을 봅니다. 여기에는 함정이 하나 있습니다. 1년차가 **둘**(최지우, 그리고 1-3 에서 넣은 한지민)이라 **동점**이 생깁니다. 동점끼리의 앞뒤는 정해져 있지 않으므로, 이름을 **보조 정렬키**로 하나 더 두어 순서를 못 박습니다.

In [49]:
# 오름차순은 기본값이라 ASC 를 적지 않아도 된다. 쉼표 뒤의 e.name 이 보조 정렬키다
# 보조 정렬키가 없으면 1년차 두 명(최지우·한지민)의 앞뒤가 실행할 때마다 달라질 수 있다
rows = run_cypher(
    "MATCH (e:Employee) RETURN e.name AS name, e.years AS years "
    "ORDER BY e.years, e.name LIMIT 3"
)
for r in rows:
    print(r['name'], r['years'])

최지우 1
한지민 1
정민재 2


> `LIMIT` 은 **줄을 세운 다음**에 자릅니다. 그래서 `ORDER BY` 를 빼면 "상위 세 명" 이 아니라 그냥 아무 세 명이 옵니다. 반대로 `ORDER BY` 만 쓰고 `LIMIT` 을 빼면 전부 다 오되 순서만 정해집니다. "상위 N" 은 **둘을 함께** 써야 나옵니다.

### 🖐️ 함께 따라하기: 학년이 높은 학생 세 명

다시 **동아리 그래프**입니다. 학년(`grade`)이 **높은 순**으로 학생 **세 명**을 뽑으세요. `RETURN s.name AS name, s.grade AS grade` 로 두 값을 받고, `ORDER BY s.grade DESC, s.name` 으로 줄을 세운 뒤 `LIMIT 3` 으로 자릅니다. 받은 순서 그대로 `for` 로 한 줄씩 출력하세요(`sorted()` 를 쓰면 애써 세운 줄이 흐트러집니다).

**확인 기준**: `노태윤 4` / `구본재 3` / `윤도현 3` 세 줄입니다. 3학년이 둘이라 동점이 생기는데, 보조 정렬키 `s.name` 이 그 둘의 앞뒤를 이름순으로 못 박습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (s:Student) 로 학생을 찾고 이름과 학년을 별칭 name·grade 로 RETURN 한다
# 2) ORDER BY s.grade DESC, s.name 으로 줄을 세운다(뒤가 보조 정렬키)
# 3) LIMIT 3 으로 위에서 세 줄만 남기고, 받은 순서 그대로 for 로 출력한다

### ✅ 바로 확인 퀴즈

**1.** `ORDER BY` 없이 `LIMIT 3` 만 쓰면 무엇을 얻게 되나요?

<details><summary>정답 보기</summary>

**아무 세 줄**입니다. 줄을 세우지 않았으니 "상위 세 개" 가 아니라 그저 먼저 걸린 세 개일 뿐이고, 실행할 때마다 달라질 수도 있습니다. "상위 N" 은 `ORDER BY` 와 `LIMIT` 을 **함께** 써야 합니다.

</details>

**2.** 근속연수가 같은 직원이 둘일 때, 실행할 때마다 같은 순서가 나오게 하려면 어떻게 하나요?

<details><summary>정답 보기</summary>

**보조 정렬키**를 하나 더 둡니다. `ORDER BY e.years, e.name` 처럼 쉼표로 이으면 근속이 같을 때 이름순으로 앞뒤가 정해져 순서가 흔들리지 않습니다.

</details>

**3.** `ORDER BY` 로 줄을 세운 결과를 파이썬에서 `sorted()` 로 한 번 더 정렬하면 어떻게 되나요?

<details><summary>정답 보기</summary>

애써 세운 줄이 **흐트러집니다.** `sorted()` 는 파이썬이 정한 기준으로 다시 줄을 세우므로 `ORDER BY DESC` 로 받아 온 내림차순이 오름차순으로 뒤집히기도 합니다. 순서가 답일 때는 받은 순서 그대로 씁니다.

</details>

---
# 4. 멱등 적재: MERGE

**멱등(idempotent)** 이란 **몇 번을 실행해도 결과가 같다**는 뜻입니다. 한 번 실행한 뒤의 상태와 열 번 실행한 뒤의 상태가 구별되지 않으면 그 연산은 멱등합니다. 엘리베이터 버튼을 여러 번 눌러도 한 번 누른 것과 같은 것처럼요.

지금까지 쓴 `CREATE` 는 **멱등하지 않습니다.** 누를 때마다 노드가 하나씩 늘어나니, 한 번 실행한 그래프와 두 번 실행한 그래프가 다릅니다. 반면 **`MERGE` 는 멱등합니다.** 이미 있으면 그대로 두기 때문입니다. 그래서 적재 코드에는 MERGE 를 씁니다.

적재 코드는 한 번만 돌지 않습니다. 실수로 두 번 누르기도 하고, 배치가 매일 돌기도 하죠. 그때마다 데이터가 불어나면 곤란합니다. 4-1 에서 **MERGE 로 노드와 관계를 멱등하게** 만들고, 4-2 에서 **MERGE 가 무엇을 "같은 것" 으로 보는지**를 보고, 4-3 에서 **처음 만들 때와 다시 만났을 때 서로 다른 값을 채우는 법**을 봅니다.

## 4-1. 노드와 관계를 MERGE 로

### 왜 필요할까요?
말로만 들으면 그럴듯하니 **같은 문장을 실제로 두 번 실행해** 눈으로 확인합니다. `CREATE` 로 쓴 것과 `MERGE` 로 쓴 것을 나란히 두 번씩 돌려 보면, 남는 노드 수가 갈립니다. 적재 코드가 어느 쪽이어야 하는지는 그 숫자가 말해 줍니다.

### 문법: MERGE
```text
MERGE (:Team {name: '인프라팀'})
```

- 이 팀이 **이미 있으면** 아무것도 하지 않고, **없으면** 만듭니다.
- 노드 개수는 조회 결과의 행 수를 **파이썬 `len()`** 으로 셉니다: `len(run_cypher("MATCH (t:Team {name: '인프라팀'}) RETURN t"))`.

In [50]:
# MERGE 는 멱등하다. 같은 노드를 두 번 MERGE 해도 하나뿐
run_cypher("MERGE (:Team {name: '인프라팀'})")   # 아직 없으니 이번엔 만든다
run_cypher("MERGE (:Team {name: '인프라팀'})")   # 다시 실행: 그래도 하나
# 개수 세기: MATCH 가 조건에 맞는 노드를 한 줄씩 돌려주므로 그 행 수가 곧 개수다
n = len(run_cypher("MATCH (t:Team {name: '인프라팀'}) RETURN t"))
print("인프라팀 노드 수(MERGE 두 번):", n)

인프라팀 노드 수(MERGE 두 번): 1


In [51]:
# 반대로 CREATE 는 실행할 때마다 새로 만든다. 중복이 쌓인다
# 이 셀은 한 번만 실행하세요. 다시 실행하면 중복팀이 3개, 4개로 계속 늘어납니다
run_cypher("CREATE (:Team {name: '중복팀'})")   # 이미 있는지 보지 않고 무조건 하나 만든다
run_cypher("CREATE (:Team {name: '중복팀'})")   # 또 만든다. 이제 두 개
n = len(run_cypher("MATCH (t:Team {name: '중복팀'}) RETURN t"))
print("중복팀 노드 수(CREATE 두 번):", n)   # 이름이 같아도 서로 다른 노드 두 개다

중복팀 노드 수(CREATE 두 번): 2


<img src="images/create-vs-merge-flow.png" width="820">

반복 적재가 예상되면 MERGE 가 기본값입니다. CREATE 는 실행한 횟수만큼 데이터가 늘어납니다.

### 문법: 관계도 MERGE 로
```text
MATCH (a:Employee {name: '...'}), (b:Team {name: '...'})
MERGE (a)-[:WORKS_IN]->(b)
```

- 노드와 똑같습니다. 양쪽 노드를 `MATCH` 로 먼저 잡고, 그 사이 화살표를 `CREATE` 대신 **`MERGE`** 로 긋습니다.
- 그 화살표가 **이미 있으면 그대로 두고**, 없을 때만 만듭니다. 그래서 같은 적재 코드를 여러 번 돌려도 **같은 관계가 두 개로 늘지 않습니다.**
- 관계 개수도 파이썬 `len()` 으로 셉니다.

In [53]:
# 관계 MERGE 도 멱등하다. 양쪽 노드를 MATCH 로 잡고 그 사이 화살표를 CREATE 대신 MERGE 로 긋는다
# 같은 쿼리를 두 번 쓰려고 문자열을 변수에 담아 둔다
rel_q = (
    "MATCH (e:Employee {name: '김서준'}), (t:Team {name: '인프라팀'}) "
    "MERGE (e)-[:WORKS_IN]->(t)"
)
run_cypher(rel_q)   # 아직 없으니 이번엔 화살표를 긋는다
run_cypher(rel_q)   # 다시 실행: 그래도 화살표는 하나
# 관계를 셀 때도 패턴을 MATCH 해 행 수를 len() 으로 센다
# RETURN 1 AS x 는 값은 필요 없고 '몇 줄 걸렸는지'만 알면 될 때 쓰는 자리표시다
n = len(run_cypher(
    "MATCH (:Employee {name: '김서준'})-[:WORKS_IN]->(:Team {name: '인프라팀'}) RETURN 1 AS x"
))
print("김서준-인프라팀 관계 수(MERGE 두 번):", n)   # 김서준은 개발팀에도 속하지만 패턴이 인프라팀을 콕 집었다

김서준-인프라팀 관계 수(MERGE 두 번): 1


> `CREATE` 를 두 번 하면 이름이 같아도 **서로 다른 노드 두 개**가 생깁니다(위에서 2). 반복 적재가 필요할 땐 노드도 관계도 반드시 **MERGE** 를 씁니다.

### 🖐️ 함께 따라하기: 같은 등록을 두 번 실행해 보기

다시 **동아리 그래프**입니다. **구본재**를 **봄전시회**에 참가시키는 문장을 `MERGE` 로 쓰되, **두 번 실행**하세요. 그런 다음 구본재와 봄전시회 사이의 `JOINS` 관계 수를 `len(run_cypher(...))` 로 세어 출력하세요.

**확인 기준**: 두 번 실행했는데도 관계는 **1** 개입니다. `CREATE` 로 썼다면 2 가 됩니다.

In [61]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 구본재와 봄전시회를 MATCH 로 찾아 그 사이 JOINS 관계를 MERGE 하는 문을 변수에 담는다
# 2) 그 문을 run_cypher 로 두 번 실행한다
# 3) 그 관계를 MATCH 해 len() 으로 세어 출력한다(1 이어야 한다)

join_q = (
    "MATCH (s:Student {name: '구본재'}), (e:Event {name: '봄전시회'}) "
    "MERGE (s)-[:JOINS]->(e)"
)
run_cypher(join_q)
run_cypher(join_q)

print(run_cypher("MATCH (s:Student {name:'구본재'})-[:JOINS]->(e) RETURN e"))

[{'e': {'name': '봄전시회', 'status': '모집중'}}, {'e': {'name': '가을공연', 'status': '마감'}}]


### ✅ 바로 확인 퀴즈

**1.** 같은 `MERGE (:Team {name: '인프라팀'})` 를 세 번 실행하면 인프라팀 노드는 몇 개가 되나요?

<details><summary>정답 보기</summary>

**한 개**입니다. MERGE 는 이미 있으면 새로 만들지 않으므로, 몇 번을 실행해도 하나뿐입니다(멱등).

</details>

**2.** 반복 적재(같은 코드를 여러 번 실행)가 예상될 때 `CREATE` 대신 `MERGE` 를 쓰는 이유는?

<details><summary>정답 보기</summary>

`CREATE` 는 실행할 때마다 새 노드·관계를 만들어 **중복이 쌓이지만**, `MERGE` 는 이미 있으면 그대로 두어 **중복이 생기지 않기** 때문입니다.

</details>

**3.** 노드만 `MERGE` 하고 관계는 `CREATE` 로 두면 재실행에서 무엇이 늘어나나요?

<details><summary>정답 보기</summary>

**관계**가 늘어납니다. 노드는 하나로 유지되지만 그 노드를 잇는 **같은 화살표가 실행할 때마다 하나씩** 더 그어집니다. 멱등하게 적재하려면 노드도 관계도 모두 MERGE 여야 합니다.

</details>

---
## 4-2. MERGE 가 보는 '같은 것'

### 왜 필요할까요?
MERGE 를 쓰기만 하면 안전한 것은 아닙니다. **무엇을 같다고 볼지**는 우리가 적은 속성이 정합니다. 여기를 잘못 적으면 MERGE 를 썼는데도 중복이 쌓입니다.

**⚠️ CREATE vs MERGE 함정**: MERGE 는 **패턴 전체가 일치**할 때만 "있다"고 봅니다. 우리 그래프의 김서준은 역할이 `백엔드` 인데, `MERGE (:Employee {name: '김서준', role: '프론트'})` 는 이름·역할이 **둘 다** 같은 노드를 찾으므로 "없다"고 보고 **김서준을 하나 더** 만들어 버립니다. 그래서 MERGE 로 콕 집을 때는 **식별 속성만**(예: `{name: ...}`) 넣는 게 안전합니다.

아래 셀에서 직접 확인해 봅니다(시드 데이터를 건드리지 않도록 새 팀 이름으로 시연합니다).

In [62]:
# MERGE 의 '같은 것' 기준은 적은 속성 전부다. 하나라도 다르면 다른 노드로 보고 새로 만든다
run_cypher("MERGE (:Team {name: '보안팀'})")   # name 만 적었다. 없으니 만든다
run_cypher("MERGE (:Team {name: '보안팀', floor: 7})")   # name 은 같지만 floor 가 더 붙었다
# 세는 조건은 name 뿐이라 위에서 만든 두 노드가 모두 걸린다
n = len(run_cypher("MATCH (t:Team {name: '보안팀'}) RETURN t"))
print("보안팀 노드 수(속성을 다르게 적은 MERGE 두 번):", n)   # MERGE 에 식별 속성만 적어야 하는 이유

보안팀 노드 수(속성을 다르게 적은 MERGE 두 번): 2


<img src="images/merge-partial-match-trap.png" width="820">

MERGE 에 무엇을 적느냐가 곧 "같은 것"의 기준입니다. 적은 속성이 하나라도 다르면 다른 노드로 봅니다.

### 관계 속성과 MERGE
교안_01 의 1-5 에서 관계에도 속성을 붙였습니다(`-[:ASSIGNED_TO {since: 2024, hours: 20}]->`). 그 속성을 `MERGE` 에 함께 적으면 어떻게 될까요. 노드와 **똑같습니다**. 적은 속성까지 "같은 것"의 기준이 되므로, 값이 하나만 달라도 "없다"고 보고 화살표를 하나 더 긋습니다.

In [63]:
# 최지우는 시드가 만든 앱개편 배정이 이미 있다(그 관계에는 속성이 없다)
# 거기에 hours 를 붙여 MERGE 하면 '속성이 hours: 10 인 배정' 은 없으므로 화살표를 하나 더 긋는다
run_cypher(
    "MATCH (e:Employee {name: '최지우'}), (p:Project {name: '앱개편'}) "
    "MERGE (e)-[:ASSIGNED_TO {hours: 10}]->(p)"
)
n = len(run_cypher(
    "MATCH (:Employee {name: '최지우'})-[r:ASSIGNED_TO]->(:Project {name: '앱개편'}) RETURN r"
))
print("최지우-앱개편 배정 관계 수:", n)   # 속성이 다르면 MERGE 는 다른 관계로 본다

최지우-앱개편 배정 관계 수: 2


> 그래서 반복 적재에서는 **MERGE 에 관계를 식별하는 부분만** 적습니다. 자주 바뀌는 값(수량·점수·갱신 시각)까지 MERGE 에 적으면 값이 바뀔 때마다 화살표가 하나씩 늘어납니다. 그런 값은 MERGE 로 찾은 **뒤에** 채워 넣어야 하는데, 그 자리가 바로 다음 절의 `ON CREATE SET`·`ON MATCH SET` 입니다.

### 🖐️ 함께 따라하기: MERGE 로 기존 노드 확인

다시 **동아리 그래프**입니다. 이미 있는 동아리 **사진동아리**를 `MERGE (:Club {name: '사진동아리'})` 로 다시 넣어 보세요. 그런 다음 사진동아리 노드를 `MATCH` 해 `len()` 으로 개수를 세어 출력하세요.

**확인 기준**: **1** 입니다. MERGE 는 이미 있으면 새로 만들지 않기 때문입니다. `CREATE` 로 썼다면 2 가 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MERGE (:Club {name: '사진동아리'}) 를 실행한다(이미 있는 동아리)
# 2) 사진동아리 노드를 MATCH 해 len() 으로 개수를 세어 출력한다. 1 이어야 한다

### ✅ 바로 확인 퀴즈

**1.** `MERGE (:Team {name: '보안팀'})` 을 실행한 뒤 `MERGE (:Team {name: '보안팀', floor: 7})` 을 실행하면 보안팀 노드는 몇 개가 되나요?

<details><summary>정답 보기</summary>

**두 개**입니다. MERGE 는 **적은 속성이 전부 같을 때만** "있다"고 봅니다. 두 번째 문은 `floor` 가 더 붙어 패턴이 다르므로 "없다"고 보고 새로 만듭니다. 그래서 MERGE 에는 **식별 속성만** 적습니다.

</details>

**2.** 배치가 매일 도는 적재 코드에서 `MERGE (a)-[:VISITED {at: 오늘날짜}]->(b)` 라고 쓰면 한 달 뒤 그 관계는 몇 개가 되어 있을까요?

<details><summary>정답 보기</summary>

**서른 개쯤**입니다. `at` 이 날마다 달라 MERGE 가 매번 "없다"고 보고 새 화살표를 긋기 때문입니다. 자주 바뀌는 값은 MERGE 에 적지 않고, 관계를 **식별하는 부분만** 적습니다.

</details>

---
## 4-3. ON CREATE SET 과 ON MATCH SET

### 왜 필요할까요?
4-2 에서 배운 규칙은 "MERGE 에는 식별 속성만 적는다" 였습니다. 그러면 나머지 값은 언제 채울까요. 게다가 값마다 사정이 다릅니다. **처음 만들 때만** 남겨야 하는 값(처음 들어온 해)이 있고, **다시 마주칠 때마다** 갱신해야 하는 값(마지막으로 확인한 횟수)이 있습니다. 같은 MERGE 문이 **처음 실행될 때와 두 번째 실행될 때 서로 다른 일**을 하게 만드는 자리가 여기입니다.

### 문법: MERGE … ON CREATE SET … ON MATCH SET
```text
MERGE (t:Team {name: '리서치팀'})
ON CREATE SET t.created_at = 2026
ON MATCH SET t.seen = 1
```

- **`ON CREATE SET`** 은 MERGE 가 **새로 만들었을 때만** 실행됩니다.
- **`ON MATCH SET`** 은 **이미 있어서 그대로 뒀을 때만** 실행됩니다.
- 둘 중 하나만 써도 되고, 둘 다 쓸 때는 `ON CREATE` 를 먼저 적습니다.
- MERGE 패턴 안에는 여전히 **식별 속성만** 넣습니다(4-2 의 규칙). 자주 바뀌는 값은 이 자리에서 채우면 화살표·노드가 늘지 않습니다.
- 값을 적어 넣는 `SET` 은 교안_01 에서 배운 그 `SET` 입니다. 앞에 붙은 `ON CREATE`·`ON MATCH` 가 **언제 실행할지**를 정할 뿐입니다.

같은 문장이 첫 실행과 두 번째 실행에서 어느 쪽으로 갈라지는지 그림으로 보면 이렇습니다.

<img src="images/on-create-match-set.png" width="820">

In [64]:
# 같은 문장을 두 번 실행해 볼 것이므로 문자열을 변수에 담아 둔다
# 패턴 안에는 식별 속성(name)만 두고, 나머지 값은 ON CREATE / ON MATCH 쪽에서 채운다
merge_q = (
    "MERGE (t:Team {name: '리서치팀'}) "
    "ON CREATE SET t.created_at = 2026 "
    "ON MATCH SET t.seen = 1"
)
run_cypher(merge_q)   # 리서치팀은 아직 없다. 새로 만들면서 ON CREATE 쪽만 실행된다
# 값이 어떻게 들어갔는지 본다. [0] 은 돌려받은 행 목록의 첫 줄이다
row = run_cypher(
    "MATCH (t:Team {name: '리서치팀'}) RETURN t.created_at AS created_at, t.seen AS seen"
)[0]
print("첫 실행 뒤:", row)   # seen 은 아직 없는 속성이라 널이다

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `seen` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=68, offset=67>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 67, 'line': 1, 'column': 68}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (t:Team {name: '리서치팀'}) RETURN t.created_at AS created_at, t.seen AS seen"


첫 실행 뒤: {'created_at': 2026, 'seen': None}


> 이 셀 아래에 Neo4j 가 **"`seen` 이라는 속성 키가 아직 없다"** 는 안내를 한 줄 띄웁니다(주피터에서는 빨간 블록으로 보입니다). **에러가 아닙니다.** 아직 아무도 그 속성을 만든 적이 없으니 널이라는 뜻이고, 바로 다음 셀에서 `ON MATCH SET` 이 값을 채우면 사라집니다.

In [67]:
# 똑같은 문장을 한 번 더 실행한다. 이번엔 리서치팀이 이미 있으므로 ON MATCH 쪽만 실행된다
run_cypher(merge_q)
row = run_cypher(
    "MATCH (t:Team {name: '리서치팀'}) RETURN t.created_at AS created_at, t.seen AS seen"
)[0]
# created_at 은 처음 값 그대로 남고 seen 만 새로 채워졌다. 두 시점이 서로를 덮어쓰지 않는다
print("두 번째 실행 뒤:", row)

두 번째 실행 뒤: {'created_at': 2026, 'seen': 1}


In [66]:
# 노드가 늘지 않았는지도 확인한다. MERGE 이므로 두 번 실행해도 하나뿐이다
n = len(run_cypher("MATCH (t:Team {name: '리서치팀'}) RETURN t"))
print("리서치팀 노드 수:", n)

리서치팀 노드 수: 1


> 이 조합이 실무에서 자주 쓰이는 이유는 분명합니다. **"언제 처음 들어왔는지" 는 지키고, "마지막으로 언제 다시 봤는지" 는 갱신**해야 하기 때문입니다. 두 값을 그냥 `MERGE` 패턴 안에 적으면 4-2 에서 본 대로 노드가 늘어나고, 둘 다 `SET` 으로만 덮어쓰면 처음 값이 사라집니다. `ON CREATE`·`ON MATCH` 로 갈라 적어야 둘 다 지켜집니다.

### 🖐️ 함께 따라하기: 처음 만든 해와 확인 표시를 갈라 적기

다시 **동아리 그래프**입니다. **요리동아리**를 `MERGE` 하되, 처음 만들 때는 `opened` 를 `2026` 으로, 이미 있을 때는 `checked` 를 `1` 로 채우도록 `ON CREATE SET`·`ON MATCH SET` 을 붙이세요. 그 문장을 **두 번 실행**한 뒤 `opened` 와 `checked` 를 함께 RETURN 해 출력합니다.

**확인 기준**: `{'opened': 2026, 'checked': 1}` 입니다. 첫 실행이 `opened` 를, 두 번째 실행이 `checked` 를 채웠고 `opened` 는 덮어쓰이지 않았습니다.

In [86]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MERGE (c:Club {name: '요리동아리'}) 에 ON CREATE SET c.opened = 2026 과
#    ON MATCH SET c.checked = 1 을 붙인 문장을 변수에 담는다(패턴엔 식별 속성 name 만)
# 2) 그 문장을 run_cypher 로 두 번 실행한다
# 3) opened 와 checked 를 함께 RETURN 해 첫 줄([0])을 출력한다

cluq_q = """
MERGE (c:Club {name:'요리동아리'})
ON CREATE SET c.opened = 2026
ON MATCH SET c.checked =  1
"""

run_cypher(cluq_q)
print(run_cypher("MATCH (c:Club {name:'요리동아리'}) RETURN c"))

run_cypher(cluq_q)
print(run_cypher("MATCH (c:Club {name:'요리동아리'}) RETURN c"))

[{'c': {'name': '요리동아리', 'checked': 1, 'opened': 2026}}]
[{'c': {'name': '요리동아리', 'checked': 1, 'opened': 2026}}]


### ✅ 바로 확인 퀴즈

**1.** `MERGE … ON CREATE SET t.created_at = 2026` 만 쓴 문장을 열 번 실행하면 `created_at` 은 어떤 값이 되나요?

<details><summary>정답 보기</summary>

**처음 실행할 때 넣은 값 그대로**입니다. 두 번째 실행부터는 노드가 이미 있어 `ON CREATE SET` 이 실행되지 않기 때문입니다. "처음 들어온 시점" 을 지키고 싶을 때 이렇게 씁니다.

</details>

**2.** 자주 바뀌는 값(예: 마지막 확인 시각)을 MERGE 패턴 안에 적으면 안 되는 이유는 무엇이고, 대신 어디에 적어야 하나요?

<details><summary>정답 보기</summary>

패턴 안에 적으면 그 값이 "같은 것" 의 기준이 되어(4-2) 값이 바뀔 때마다 **새 노드·관계가 생깁니다.** 패턴에는 식별 속성만 두고, 그런 값은 **`ON MATCH SET`**(또는 `ON CREATE SET`) 에 적어야 합니다.

</details>

**3.** `ON CREATE SET` 과 `ON MATCH SET` 은 한 문장에서 몇 개가 실행되나요?

<details><summary>정답 보기</summary>

**둘 중 하나만** 실행됩니다. MERGE 가 새로 만들었으면 `ON CREATE` 쪽이, 이미 있어 그대로 뒀으면 `ON MATCH` 쪽이 실행됩니다. 그래서 첫 실행과 두 번째 실행의 결과가 달라집니다.

</details>

---
# 5. 파이썬 값을 쿼리에 넘기기

지금까지 쿼리 안의 값은 전부 **글자 그대로**(`{name: '김서준'}`) 적었습니다. 그런데 적재할 값이 파이썬 리스트에 들어 있다면 어떻게 할까요. 이름이 바뀔 때마다 쿼리 문자열을 새로 조립하는 것은 번거롭고 위험합니다.

## 문법: `$이름` 자리표시자
쿼리 안에는 값 대신 **`$이름`** 이라는 자리표시자를 두고, 실제 값은 `run_cypher` 의 **이름 붙인 인자**로 따로 넘깁니다.

```text
run_cypher("MERGE (:Team {name: $team})", team='인프라팀')
                                    ↑ 자리표시자        ↑ 여기로 값이 들어간다
```

값이 따로 가므로 따옴표를 맞출 필요가 없고, 리스트를 `for` 로 돌며 같은 쿼리를 재사용할 수 있습니다. 맨 위 시드 셀도 이 방식으로 데이터를 넣었습니다.

값이 어디로 들어가는지, 그리고 이어 붙였을 때 무엇이 깨지는지 그림으로 보면 이렇습니다.

<img src="images/param-placeholder.png" width="820">

In [90]:
# 파이썬 리스트에 든 팀 이름들을 같은 쿼리 하나로 적재한다
# 쿼리 문자열은 그대로 두고 team 자리에 들어갈 값만 매번 바뀐다
new_teams = ['인프라팀', '보안팀', '데이터팀', '노래방팀']
for team in new_teams:
    run_cypher("MERGE (:Team {name: $test})", test=team)   # MERGE 라 여러 번 돌려도 늘지 않는다
rows = run_cypher("MATCH (t:Team) RETURN t.name AS name")
# 인프라팀·보안팀은 4장 시연에서 이미 만들어 MERGE 가 그대로 뒀고, 새로 생긴 건 데이터팀 하나다
# 중복팀 둘·보안팀 둘은 4-1 의 CREATE 시연과 4-2 의 속성을 다르게 적은 MERGE 가 남긴 자국이다
print('팀 목록:', sorted(r['name'] for r in rows))

팀 목록: ['개발팀', '노래방팀', '데이터팀', '디자인팀', '리서치팀', '보안팀', '보안팀', '인프라팀', '중복팀', '중복팀']


> 값을 **문자열에 이어 붙이지 않는** 것이 핵심입니다. `"MERGE (:Team {name: '" + team + "'})"` 처럼 조립하면 이름에 따옴표가 들어 있을 때 쿼리가 깨지고, 남이 넣은 값이 쿼리를 바꿔치기할 수도 있습니다. 자리표시자를 쓰면 값은 **값으로만** 전달됩니다.

### 🖐️ 함께 따라하기: 동아리를 파라미터로 적재하기

다시 **동아리 그래프**입니다. 파이썬 리스트 `['보드게임동아리', '사진동아리', '요리동아리']` 를 `for` 로 돌며 `$name` 자리표시자로 `Club` 노드를 `MERGE` 하세요. 그다음 동아리 이름을 정렬해 출력합니다.

**확인 기준**: 시드에 있던 사진동아리와 4-3 에서 만든 요리동아리는 **이미 있으니 그대로**고, 보드게임동아리 **하나가 새로 생깁니다**. 밴드동아리까지 더해 결과는 `['밴드동아리', '보드게임동아리', '사진동아리', '요리동아리']` **네 개**입니다.

In [92]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 동아리 이름 세 개를 파이썬 리스트로 적어 둔다
# 2) for 로 돌며 MERGE (:Club {name: $name}) 를 실행하고 값은 name= 인자로 넘긴다
# 3) Club 이름을 전부 조회해 정렬 출력한다

news = ['보드게임동아리', '사진동아리', '요리동아리']
for name in news:
    run_cypher("MERGE (:Club {name: $test})", test=name)   # MERGE 라 여러 번 돌려도 늘지 않는다
rows = run_cypher("MATCH (c:Club) RETURN c.name AS name")

print('동아리 목록:', sorted(r['name'] for r in rows))

동아리 목록: ['밴드동아리', '보드게임동아리', '사진동아리', '요리동아리']


### ✅ 바로 확인 퀴즈

**1.** `run_cypher("MERGE (:Team {name: $team})", team='보안팀')` 에서 `$team` 자리에는 무엇이 들어가나요?

<details><summary>정답 보기</summary>

**`team=` 으로 넘긴 값**(`'보안팀'`)이 들어갑니다. 쿼리 문자열은 그대로 두고 값만 따로 전달됩니다.

</details>

**2.** 값을 자리표시자로 넘기지 않고 문자열에 이어 붙이면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

이름에 **따옴표**가 들어 있으면 쿼리가 깨지고, 값이 쿼리 문법으로 읽혀 **엉뚱한 쿼리로 바뀔** 수 있습니다. 자리표시자를 쓰면 값은 값으로만 전달돼 그런 일이 없습니다.

</details>

---
## 🚀 응용 클론코딩: 조건으로 찾은 동료를 멱등하게 팀에 넣기

1. **근속 4년 이상**인 직원의 이름을 `WHERE e.years >= 4` 로 찾아 정렬 출력한다(별칭 `name`).
2. 새 팀 **시니어모임**을 `MERGE` 로 만들되, 처음 만들 때만 `founded` 를 `2026` 으로 채운다(`ON CREATE SET`).
3. 같은 `MERGE` 를 한 번 더 실행해도 시니어모임이 **하나뿐**인지 `len()` 으로 확인해 출력한다.

In [104]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) WHERE e.years >= 4 로 직원 이름을 찾아(별칭 name) 정렬 출력한다
# 2) MERGE (t:Team {name: '시니어모임'}) ON CREATE SET t.founded = 2026 을 두 번 실행한다
# 3) 시니어모임 노드를 MATCH 해 len() 으로 개수를 세어 출력한다(1 이어야 함)

se = run_cypher("MATCH (e:Employee) WHERE e.years >=4 RETURN e.name AS name")
print(se)

senior_q = """
MERGE (t:Team {name:'시니어모임'}) ON CREATE SET t.founded = 2026
"""
run_cypher(senior_q)
run_cypher(senior_q)

print(run_cypher("MATCH (t:Team {name:'시니어모임'}) RETURN count(t) AS count"))


# 4) 찾은 시니어를 팀에 배정
for senior in se :
    name = senior['name']
    run_cypher("MERGE (e:Employee {name:$name})-[:WORKS_IN]->(t:Team {name:'시니어모임'})", name=name)

print(run_cypher("MERGE (e:Employee)-[:WORKS_IN]->(t:Team {name:'시니어모임'}) RETURN e"))

run_cypher("""
MATCH (e:Employee), (t:Team {name:'시니어모임'})
WHERE e.years >= 4
MERGE (e)-[:WORK_IN]->(t)
""")


print(run_cypher("MERGE (e:Employee)-[:WORKS_IN]->(t:Team {name:'시니어모임'}) RETURN e"))





[{'name': '김서준'}, {'name': '박도윤'}]
[{'count': 5}]
[{'e': {'name': '김서준'}}, {'e': {'name': '박도윤'}}]
[{'e': {'name': '김서준'}}, {'e': {'name': '박도윤'}}]


In [107]:
# WORK_IN 관계만 깔끔하게 청소 삭제!
run_cypher("MATCH ()-[r:WORK_IN]->() DELETE r")
print("오타 관계(WORK_IN) 청소 완료!")


오타 관계(WORK_IN) 청소 완료!


In [108]:
# 현재 DB에 남아있는 관계 종류 확인
types = run_cypher("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType")
print("현재 관계 종류:", [r['relationshipType'] for r in types])


현재 관계 종류: ['OWNS', 'WORKS_IN', 'ASSIGNED_TO', 'HOSTS', 'BELONGS_TO', 'JOINS']


---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-1 | `WHERE e.years >= 3` | 비교(`>=`,`<=`,`<>`)를 건다 |
| 1-2 | `WHERE ... AND ... / OR ...` | 둘 다 / 하나라도 |
| 1-3 | `WHERE NOT e.role = '백엔드'` | 뒤의 조건을 통째로 뒤집는다(`<>` 와 같은 뜻) |
| 1-3 | `WHERE e.role IS NULL` | 속성이 아예 없는 노드를 찾는다 |
| 2-1 | `(a)-[:R]->(b)-[:S]->(c)` | 관계를 이어 거쳐가며 따라간다 |
| 2-1 | `(b)<-[:R]-(a)` | 화살표를 거꾸로 읽어 거슬러 간다 |
| 2-2 | `(a)-[:R]->(b)<-[:S]-(c)` | 가운데 노드를 공유해 연결한다 |
| 3-1 | `MATCH (p)<-[:R]-(e) RETURN p.name` | 노드가 아니라 **패턴이 맞은 경우**를 센다 |
| 3-2 | `RETURN DISTINCT p.name` | 되풀이를 DB 가 접는다(행 전체가 기준) |
| 3-3 | `ORDER BY e.years DESC LIMIT 3` | 줄을 세운 뒤 위에서 몇 개만 받는다 |
| 4-1 | `MERGE (:레이블 {속성})` | 있으면 그대로, 없으면 만든다 |
| 4-1 | `MATCH (a),(b) MERGE (a)-[:R]->(b)` | 같은 화살표가 두 개로 늘지 않는다 |
| 4-2 | `MERGE` 에 적은 속성 | 그것이 곧 "같은 것" 의 기준이다 |
| 4-3 | `ON CREATE SET` / `ON MATCH SET` | 처음 만들 때와 다시 만났을 때를 가른다 |
| 5 | `run_cypher("... $이름 ...", 이름=값)` | 쿼리는 그대로, 값만 따로 전달한다 |

- 속성 map 은 **정확히 같은 값**, WHERE 는 **비교·조합·부정** 조건.
- 속성이 비어 있는 노드는 `=` 에도 `NOT =` 에도 걸리지 않습니다. 그런 노드는 **`IS NULL`** 로 따로 찾습니다.
- 화살표를 거꾸로(`<-`) 그려도 관계의 **종류와 방향은 그대로**입니다. 읽는 출발점만 바뀝니다.
- 같은 이름이 여러 번 나오면 버그가 아니라 **길이 여럿**이라는 뜻입니다. **`DISTINCT`** 로 DB 에서 접거나 파이썬 집합으로 접습니다.
- "상위 N" 은 **`ORDER BY` 와 `LIMIT` 을 함께** 써야 나옵니다. 동점이 있으면 **보조 정렬키**를 하나 더 둡니다.
- 반복 적재에는 중복을 막는 **MERGE** 를 씁니다. MERGE 에는 **식별 속성만** 적고, 나머지 값은 **`ON CREATE SET`·`ON MATCH SET`** 으로 갈라 채웁니다.
- 노드·관계 개수 확인은 파이썬 **`len(run_cypher(...))`** 으로.
- 파이썬 값은 **`$이름`** 자리표시자로 넘깁니다. 문자열에 이어 붙이면 따옴표에서 깨집니다.

## ⏭️ 예고: 다음 시간

이제 노드·관계를 만들고 조건으로 찾고, 결과를 다듬고, 멱등하게 적재하는 법을 익혔습니다. 다음 단원에서는 **경로 탐색**(몇 단계 떨어져 있는지 모를 때 쓰는 가변 길이 패턴, 최단 경로)과 **중간 결과를 이어받아 다음 단계로 넘기는 문법**을 다룹니다.

수고하셨습니다!